# Importing Libraries

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import DataFrame

In [0]:
CATALOG      = "workspace"
BRONZE_TABLE = f"{CATALOG}.sme_risk_bronze.asic_companies_raw"
SILVER_TABLE = f"{CATALOG}.sme_risk_silver.companies"

RENAME_MAP = {
    "company_name":                   "company_name",
    "acn":                            "acn",
    "type":                           "company_type",
    "class":                          "company_class",
    "sub_class":                      "company_sub_class",
    "status":                         "status_code",
    "date_of_registration":           "registration_date",
    "date_of_deregistration":         "deregistration_date",
    "previous_state_of_registration": "previous_state",
    "state_registration_number":      "state_registration_number",
    "modified_since_last_report":     "modified_since_last_report",
    "current_name_indicator":         "current_name_indicator",
    "abn":                            "abn",
    "current_name":                   "current_name",
    "current_name_start_date":        "current_name_start_date",
}

# Code labels: ASIC Company Dataset Help File, Appendix Item 2
TYPE_LABELS = {
    "APTY": "Australian proprietary company", "APUB": "Australian public company",
    "FNOS": "Foreign company registered in Australia", "RACN": "Registered Australian corporation",
    "CCIV": "Corporate collective investment vehicle (not in ASIC help file)",
}
CLASS_LABELS = {
    "LMSH": "Limited by shares", "LMGT": "Limited by guarantee", "LMSG": "Limited by shares and guarantee",
    "NLIA": "No liability", "UNLM": "Unlimited", "NONE": "No Australian equivalent",
}
SUB_CLASS_LABELS = {
    "EXPT": "Exempt proprietary", "NEXT": "Non-exempt proprietary", "PROP": "Proprietary other",
    "PNPC": "Proprietary non-profit", "PSTC": "Proprietary superannuation trustee", "HUNT": "Proprietary home unit",
    "LIST": "Listed public", "ULST": "Unlisted public", "ULSN": "Unlisted public non-profit",
    "ULSS": "Unlisted public superannuation trustee", "NLTD": "Non-profit public without 'Limited'",
    "LISN": "Licensed to omit 'Limited'", "LISS": "Licensed to omit 'Limited' (super trustee)",
    "PUBF": "Foreign company lodging balance sheet", "RACA": "Registrable Australian corporation (association)",
    "RACO": "Registrable Australian corporation (non-association)",
    "STFI": "Public small transferring financial institution", "NONE": "Unknown",
}
STATUS_LABELS = {
    "REGD": "Registered", "DRGD": "Deregistered", "EXAD": "External administration",
    "SOFF": "Strike-off action in progress", "NOAC": "Not active", "NRGD": "Not registered",
    "PROV": "Provisional", "DISS": "Dissolved by Act of Parliament", "DIV3": "Transferred via Div 3",
    "PEND": "Pending (schemes)", "CNCL": "Cancelled / transferred to another law",
}
SUB_CLASS_LABELS["WHSL"] = "Wholesale CCIV (not in ASIC help file; inferred)"

PREV_STATE_LABELS = {
    "NSW": "New South Wales", "VIC": "Victoria", "QLD": "Queensland",
    "WA": "Western Australia", "SA": "South Australia", "TAS": "Tasmania",
    "ACT": "Australian Capital Territory", "NT": "Northern Territory",
    "NFK": "Norfolk Island",
}
BRONZE_TABLE = "workspace.sme_risk_bronze.asic_companies_raw"
SILVER_TABLE = "workspace.sme_risk_silver.companies"

# Reading From the Bronze Table

In [0]:
''' Select the latest snapshot
Bronze keeps every weekly ASIC file (one full copy of the register per `_snapshot_date`).
Silver describes companies as they are now, so only the most recent snapshot is used.
Without this filter, each company would appear once per week loaded and every count downstream would be inflated.
With a single snapshot loaded, this step changes nothing; it keeps the weekly pipeline correct as history builds up.''' 

bronze = spark.table(BRONZE_TABLE)
# Bronze keeps every weekly snapshot; Silver is rebuilt from the latest one.
snapshot_date = bronze.agg(F.max("_snapshot_date")).first()[0]

df = (bronze
      .filter(F.col("_snapshot_date") == F.lit(snapshot_date))
      .drop("_source_file", "_source_url", "_ingested_at"))

baseline_rows = df.count()
baseline_acns = df.select("acn").distinct().count()
print(f"Snapshot {snapshot_date}: {baseline_rows:,} rows, {baseline_acns:,} distinct ACNs")

# Data Transformations

In [0]:
for old, new in RENAME_MAP.items():
    df = df.withColumnRenamed(old, new)

### Transform: `company_name`
Decisions are based on profiling of the 2026-10-12 snapshot and checks against ASIC's online register:
- **CSV escaping decoded:** 429 names were wrapped in quotes with inner quotes doubled
  (e.g. `"GO ""N"" OFF PTY LTD"` → `GO "N" OFF PTY LTD`, verified on ASIC for ACNs 106 366 090 and 101 413 405).
- **Formatting cleaned:** 45,798 names with leading/trailing spaces and 46,776 with double spaces.
- **Kept unchanged:** asterisks (genuine, e.g. `CREATE*IT PTY LTD`), numbers in names, case, punctuation.
- **Flagged, not changed:** `°`, which appears to replace `|` in ASIC's export (PILOTPATH example); pending further evidence.
- **New feature:** `is_acn_named`, true when the company's own ACN appears in its name (e.g. `ACN 094 072 214 PTY LTD`, or an ABN-style name containing the ACN).

In [0]:
def unescape_csv(col_name):
    """Remove CSV outer quotes and turn doubled quotes back into single ones."""
    c = F.col(col_name)
    return (F.when(c.startswith('"') & c.endswith('"') & (F.length(c) >= 2),
                   F.regexp_replace(F.expr(f"substring({col_name}, 2, length({col_name}) - 2)"), '""', '"'))
             .otherwise(c))

def clean_text(col_name):
    """Strip control characters, trim, collapse repeated spaces, blank → null."""
    c = F.regexp_replace(F.col(col_name), r"[\x00-\x1F\x7F]", "")
    c = F.regexp_replace(F.trim(c), r"\s{2,}", " ")
    return F.when(c == "", None).otherwise(c)

# 1) Undo CSV escaping on the two name columns
df = (df.withColumn("company_name", unescape_csv("company_name"))
        .withColumn("current_name", unescape_csv("current_name")))

# 2) Clean every string column (fixes name spaces, previous_state trailing spaces, etc.)
string_cols = {f.name for f in df.schema.fields if f.dataType.simpleString() == "string"}
df = df.select([clean_text(c).alias(c) if c in string_cols else F.col(c) for c in df.columns])

# 3) Name-based flags
df = (df
    .withColumn("company_name_has_export_artifact", F.coalesce(F.col("company_name").contains("°"), F.lit(False)))
    .withColumn("is_acn_named", F.coalesce(
        F.instr(F.regexp_replace(F.col("company_name"), r"[^0-9]", ""), F.col("acn")) > 0, F.lit(False))))

In [0]:
def parse_ddmmyyyy(col_name):
    return F.expr(f"try_to_timestamp({col_name}, 'dd/MM/yyyy')").cast("date")

df = (df
    # flags first, while the raw text is still there
    .withColumn("registration_date_unparseable",
                F.col("registration_date").isNotNull() & parse_ddmmyyyy("registration_date").isNull())
    .withColumn("deregistration_date_unparseable",
                F.col("deregistration_date").isNotNull() & parse_ddmmyyyy("deregistration_date").isNull())
    # dates
    .withColumn("registration_date", parse_ddmmyyyy("registration_date"))
    .withColumn("deregistration_date_reported", parse_ddmmyyyy("deregistration_date"))
    .drop("deregistration_date", "current_name_start_date")
    # ABN: "0" means no ABN
    .withColumn("abn", F.when(F.col("abn") == "0", None).otherwise(F.col("abn")))
    # Y → true, blank → false (plain == 'Y' would give null)
    .withColumn("modified_since_last_report",
                F.coalesce(F.col("modified_since_last_report") == "Y", F.lit(False)))
)

df.createOrReplaceTempView("v_companies_clean")

In [0]:
silver = spark.sql("""
WITH name_history AS (
    SELECT acn, COUNT_IF(current_name_indicator IS NULL) AS former_name_count
    FROM v_companies_clean
    GROUP BY acn
),
current_rows AS (
    SELECT * FROM v_companies_clean WHERE current_name_indicator = 'Y'
)
SELECT
    c.acn,
    c.company_name,
    c.company_type,
    c.company_class,
    c.company_sub_class,
    c.status_code,
    c.registration_date,
    CASE WHEN c.status_code = 'DRGD' THEN c.deregistration_date_reported END AS deregistration_date,
    c.deregistration_date_reported,
    c.previous_state,
    c.state_registration_number,
    c.abn,
    c.abn IS NOT NULL                                        AS has_abn,
    COALESCE(h.former_name_count, 0)                         AS former_name_count,
    c.is_acn_named,
    c.status_code = 'DRGD'                                   AS is_deregistered,
    c.status_code IN ('REGD', 'SOFF', 'EXAD', 'NOAC')        AS is_active,
    c.status_code IN ('SOFF', 'EXAD')                        AS is_distressed,
    c.status_code NOT IN ('CNCL', 'DISS')                    AS in_population,
    c.status_code <> 'DRGD' AND c.deregistration_date_reported IS NOT NULL
                                                             AS dereg_date_status_conflict,
    c.company_name_has_export_artifact,
    c.modified_since_last_report,
    c._snapshot_date
FROM current_rows c
LEFT JOIN name_history h ON c.acn = h.acn
""")

In [0]:
def label(col_name, mapping):
    m = F.create_map(*[F.lit(x) for kv in mapping.items() for x in kv])
    return F.try_element_at(m, F.col(col_name))   # null if code is unknown (caught in Cell 10)

silver = (silver
    .withColumn("company_type_label",      label("company_type", TYPE_LABELS))
    .withColumn("company_class_label",     label("company_class", CLASS_LABELS))
    .withColumn("company_sub_class_label", label("company_sub_class", SUB_CLASS_LABELS))
    .withColumn("status_label",            label("status_code", STATUS_LABELS))
    .withColumn("previous_state_label",    label("previous_state", PREV_STATE_LABELS))
)

# final column order
silver = silver.select(
    "acn", "company_name",
    "company_type", "company_type_label", "company_class", "company_class_label",
    "company_sub_class", "company_sub_class_label", "status_code", "status_label",
    "registration_date", "deregistration_date", "deregistration_date_reported",
    "previous_state", "previous_state_label", "state_registration_number",
    "abn", "has_abn", "former_name_count", "is_acn_named",
    "is_deregistered", "is_active", "is_distressed", "in_population",
    "dereg_date_status_conflict", "company_name_has_export_artifact",
    "modified_since_last_report", "_snapshot_date",
)

In [0]:
fails = {}

# --- checks on the cleaned (pre-dedup) rows ---
clean = spark.table("v_companies_clean")
no_y = (clean.groupBy("acn")
    .agg(F.max(F.coalesce((F.col("current_name_indicator") == "Y").cast("int"), F.lit(0))).alias("has_y"),
         F.max(F.when(F.coalesce(F.col("status_code"), F.lit("")) != "CNCL", 1).otherwise(0)).alias("non_cncl"))
    .filter("has_y = 0"))
no_y_acns = no_y.count()
fails["no_current_name_row_non_cncl"] = no_y.filter("non_cncl = 1").count()

pre = clean.agg(
    F.count_if("registration_date_unparseable").alias("reg_unparseable"),
    F.count_if("deregistration_date_unparseable").alias("dereg_unparseable"),
).first().asDict()
fails.update(pre)

# --- checks on the Silver table ---
acn, abn = F.col("acn"), F.col("abn")
acn_sum = sum(F.substring(acn, i, 1).cast("int") * (9 - i) for i in range(1, 9))
acn_check_ok = ((10 - acn_sum % 10) % 10) == F.substring(acn, 9, 1).cast("int")
abn_sum = sum((F.substring(abn, i, 1).cast("int") - (1 if i == 1 else 0)) * w
              for i, w in enumerate([10, 1, 3, 5, 7, 9, 11, 13, 15, 17, 19], start=1))

post = silver.agg(
    F.count("*").alias("rows"),
    F.countDistinct("acn").alias("distinct_acns"),
    F.count_if(~acn.rlike(r"^[0-9]{9}$") | acn.isNull()).alias("acn_bad_format"),
    F.count_if(acn.rlike(r"^[0-9]{9}$") & ~acn_check_ok).alias("acn_bad_check_digit"),
    F.count_if(F.col("company_type").isNotNull() & F.col("company_type_label").isNull()).alias("unknown_type"),
    F.count_if(F.col("company_class").isNotNull() & F.col("company_class_label").isNull()).alias("unknown_class"),
    F.count_if(F.col("company_sub_class").isNotNull() & F.col("company_sub_class_label").isNull()).alias("unknown_sub_class"),
    F.count_if(F.col("status_code").isNull() | F.col("status_label").isNull()).alias("unknown_status"),
    F.count_if(F.col("previous_state").isNotNull() & F.col("previous_state_label").isNull()).alias("unknown_previous_state"),
    F.count_if(F.col("is_deregistered") & F.col("deregistration_date").isNull()).alias("drgd_without_date"),
    F.count_if(F.col("registration_date") > F.col("_snapshot_date")).alias("reg_after_snapshot"),
    F.count_if(F.col("deregistration_date_reported") > F.col("_snapshot_date")).alias("dereg_after_snapshot"),
    F.count_if(F.col("deregistration_date") < F.col("registration_date")).alias("dereg_before_reg"),
    F.count_if(abn.isNotNull() & ~abn.rlike(r"^[0-9]{11}$")).alias("abn_bad_format"),
    F.count_if(abn.rlike(r"^[0-9]{11}$") & (abn_sum % 89 != 0)).alias("abn_bad_check_digit"),
    (F.count(abn) - F.countDistinct(abn)).alias("abn_duplicates"),
).first().asDict()

rows, distinct_acns = post.pop("rows"), post.pop("distinct_acns")
fails.update(post)
fails["duplicate_acns"] = rows - distinct_acns
fails["row_count_mismatch"] = abs(rows - (baseline_acns - no_y_acns))

print(f"Silver rows: {rows:,} | baseline ACNs: {baseline_acns:,} | ACNs without a current-name row: {no_y_acns:,}")
for k, v in fails.items():
    print(f"{'✅' if v == 0 else '❌'} {k}: {v:,}")

# info only (not hard stops)
display(silver.agg(
    F.min("deregistration_date").alias("window_start"), F.max("deregistration_date").alias("window_end"),
    F.count_if("is_deregistered").alias("deregistered"), F.count_if("dereg_date_status_conflict").alias("status_date_conflicts"),
    F.count_if(F.col("registration_date").isNull()).alias("missing_reg_date"), F.count_if("has_abn").alias("with_abn"),
    F.count_if("modified_since_last_report").alias("modified"), F.count_if("is_acn_named").alias("acn_named")))

bad = {k: v for k, v in fails.items() if v != 0}
assert not bad, f"Silver validation failed: {bad}"

In [0]:
(silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE))

print(f"Written {spark.table(SILVER_TABLE).count():,} rows to {SILVER_TABLE} (snapshot {snapshot_date})")

In [0]:
spark.sql(f"""COMMENT ON TABLE {SILVER_TABLE} IS
'One row per company from the latest ASIC Company Dataset snapshot. Cleaned, typed, deduplicated to the current-name row, with labels and business flags. Rebuilt in full each run.'""")

COLUMN_COMMENTS = {
    "acn": "9-digit ACN (ARBN for registrable bodies). Text to keep leading zeros. Validated incl. check digit.",
    "company_name": "Current registered name. CSV escaping removed, trimmed, repeated spaces collapsed.",
    "company_type": "ASIC company type code (APTY, APUB, FNOS, RACN, CCIV).",
    "company_class": "ASIC liability class code. NONE for foreign and registrable bodies.",
    "company_sub_class": "ASIC sub-class code (e.g. PROP, PSTC super trustee, ULSN not-for-profit).",
    "status_code": "ASIC status code (REGD, DRGD, SOFF, EXAD, NOAC, DISS). Authoritative over dates.",
    "registration_date": "Date registered. 4 missing. May be date joined the ASIC register for some post-1991 transfers.",
    "deregistration_date": "Deregistration date, filled only when status is DRGD. File holds past-year deregistrations only.",
    "deregistration_date_reported": "Deregistration date as reported in the file, regardless of status.",
    "previous_state": "State of registration before the 1991 national scheme. Not current location.",
    "state_registration_number": "Original state registry number. May contain letters.",
    "abn": "ABN as recorded by ASIC. Null when none recorded. Not a check of ABN active status.",
    "has_abn": "True when an ABN is recorded. Indicates trading vs dormant; compare within segment only.",
    "former_name_count": "Number of former-name rows for this ACN.",
    "is_acn_named": "True when the company own ACN appears in its name (numbered company).",
    "is_deregistered": "Status is DRGD.",
    "is_active": "Status is REGD, SOFF, EXAD or NOAC.",
    "is_distressed": "Status is SOFF (strike-off action) or EXAD (external administration).",
    "in_population": "Included in analysis (excludes CNCL and DISS).",
    "dereg_date_status_conflict": "Deregistration date present but status is not DRGD (likely reinstatement).",
    "company_name_has_export_artifact": "Name contains a suspected export character replacement (degree sign).",
    "modified_since_last_report": "Changed since the previous ASIC weekly file. Pipeline use only.",
    "_snapshot_date": "ASIC file snapshot date this row comes from.",
}
for col, text in COLUMN_COMMENTS.items():
    spark.sql(f"ALTER TABLE {SILVER_TABLE} ALTER COLUMN {col} COMMENT '{text}'")
for col in ["company_type", "company_class", "company_sub_class", "status", "previous_state"]:
    name = f"{col}_label"
    spark.sql(f"ALTER TABLE {SILVER_TABLE} ALTER COLUMN {name} COMMENT 'Description of {col.replace('status', 'status_code')} from the ASIC help file'")

In [0]:


cn = F.col("company_name")
cur = F.col("current_name_indicator") == "Y"

display(df.agg(
    F.sum((cn != F.trim(cn)).cast("int")).alias("leading_trailing_spaces"),      # expect 0
    F.sum(cn.rlike(r"\s{2,}").cast("int")).alias("double_spaces"),                # expect 0
    F.sum(cn.contains('""').cast("int")).alias("doubled_quotes_left"),            # expect 0
    F.sum(cn.isNull().cast("int")).alias("null_names"),                           # expect 0
    F.sum((cur & F.col("company_name_has_export_artifact")).cast("int")).alias("current_names_with_°"),   # expect 1 (PILOTPATH)
    F.sum((cur & F.col("is_acn_named")).cast("int")).alias("current_acn_named"),  # expect ≥ 12,524
))

# Spot-check the verified examples
display(df.filter(F.col("acn").isin("106366090", "101413405", "097396346", "696299475", "097927029") & cur)
          .select("acn", "company_name", "is_acn_named", "company_name_has_export_artifact"))

# Display all rows in df
display(df)

In [0]:

cn = F.col("company_name")

profile = df.agg(
    F.count("*").alias("rows"),
    F.sum(cn.isNull().cast("int")).alias("null"),
    F.sum((F.trim(cn) == "").cast("int")).alias("blank_only_spaces"),
    F.sum((cn != F.trim(cn)).cast("int")).alias("leading_trailing_spaces"),
    F.sum(cn.rlike(r"\s{2,}").cast("int")).alias("double_spaces"),
    F.sum(cn.rlike(r"[a-z]").cast("int")).alias("has_lowercase"),
    F.sum(cn.rlike(r"[\x00-\x1F\x7F]").cast("int")).alias("control_chars"),
    F.sum(cn.rlike(r"[^\x00-\x7F]").cast("int")).alias("non_ascii"),
    F.sum(cn.contains("\uFFFD").cast("int")).alias("encoding_replacement_char"),
    F.sum(cn.contains('"').cast("int")).alias("has_double_quote"),
    F.sum(cn.contains("*").cast("int")).alias("has_asterisk"),
    F.sum(cn.rlike(r"\d{3} ?\d{3} ?\d{3}").cast("int")).alias("contains_9_digit_number"),
    F.min(F.length(cn)).alias("min_len"),
    F.max(F.length(cn)).alias("max_len"),
).first().asDict()

for k, v in profile.items():
    print(f"{k:28} {v:,}")

In [0]:
display(df.filter(F.col("company_name").contains('"'))
          .select("acn", "company_name", "status_code", "current_name_indicator")
          .limit(30))

quotes = F.length("company_name") - F.length(F.regexp_replace("company_name", '"', ""))
display(df.filter(F.col("company_name").contains('"'))
          .withColumn("quote_count", quotes)
          .groupBy((F.col("quote_count") % 2 == 1).alias("unbalanced")).count())

In [0]:
def unescape_csv(col):
    c = F.col(col)
    return (F.when(c.startswith('"') & c.endswith('"') & (F.length(c) >= 2),
                   F.regexp_replace(F.expr(f"substring({col}, 2, length({col}) - 2)"), '""', '"'))
             .otherwise(c))

df = (df
    .withColumn("company_name", unescape_csv("company_name"))
    .withColumn("current_name", unescape_csv("current_name"))   # same export, same issue likely
)

# Verify: no doubled quotes left, and quotes in names now ~429 single pairs
display(df.agg(
    F.sum(F.col("company_name").contains('""').cast("int")).alias("doubled_quotes_left"),
    F.sum(F.col("company_name").startswith('"').cast("int")).alias("names_starting_with_quote"),
))

display(df.filter(F.col("company_name").contains('"'))
          .select("acn", "company_name", "status_code", "current_name_indicator")
          .limit(30))

In [0]:
cn = F.col("company_name")
cols = ["acn", "company_name", "status_code", "current_name_indicator"]

# 1) The 4 non-ASCII names, plus the exact unusual characters they contain
display(df.filter(cn.rlike(r"[^\x00-\x7F]"))
          .withColumn("non_ascii_chars", F.regexp_replace(cn, r"[\x00-\x7F]", ""))
          .select(*cols, "non_ascii_chars"))

display(df.filter(cn.contains("*"))
          .groupBy("status_code", "current_name_indicator").count())   # where do asterisks appear?

# 2) Asterisk examples: current-name rows (80)
display(df.filter(cn.contains("*") & (F.col("current_name_indicator") == "Y"))
          .select(*cols).limit(20))

# 3) Asterisk examples: historical-name rows (41)
display(df.filter(cn.contains("*") & F.col("current_name_indicator").isNull())
          .select(*cols).limit(20))

# 4) Where does the asterisk sit in the name?
display(df.filter(cn.contains("*"))
          .withColumn("asterisk_position",
              F.when(F.trim(cn).endswith("*"), "end")
               .when(F.trim(cn).startswith("*"), "start")
               .otherwise("middle"))
          .groupBy("asterisk_position", "current_name_indicator").count()
          .orderBy("asterisk_position"))

nine = (df
    .filter(cn.rlike(r"\d{3}[\s.\-]?\d{3}[\s.\-]?\d{3}"))
    # First 9-digit number in the name, separators removed (handles "004 435 652", "004435652", "004-435-652")
    .withColumn("name_number",
        F.regexp_replace(F.regexp_extract(cn, r"(\d{3}[\s.\-]?\d{3}[\s.\-]?\d{3})", 1), r"[\s.\-]", ""))
    .withColumn("match_type",
        F.when(F.col("name_number") == F.col("acn"), "own_acn").otherwise("other_number"))
    .withColumn("row_type",
        F.when(F.col("current_name_indicator") == "Y", "current_name").otherwise("historical_name"))
)

# 1) Own ACN vs other number, split by current vs historical rows
display(nine.groupBy("match_type", "row_type").count().orderBy("match_type", "row_type"))

# 2) What reaches Silver: current-name rows named after their own ACN, by status
display(nine.filter((F.col("match_type") == "own_acn") & (F.col("row_type") == "current_name"))
            .groupBy("status_code").count().orderBy(F.desc("count")))

# 3) Examples of "other_number": what are these numbers?
display(nine.filter(F.col("match_type") == "other_number")
            .select("acn", "company_name", "name_number", "status_code", "row_type").limit(20))

# 4) Examples of "own_acn" to confirm the pattern
display(nine.filter(F.col("match_type") == "own_acn")
            .select("acn", "company_name", "status_code", "row_type").limit(10))


In [0]:
%sql
SELECT company_name,current_name
FROM workspace.sme_risk_bronze.asic_companies_raw
WHERE company_name = 'VEHICLE AIR CONDITIONING SPECIALISTS OF AUSTRALASIA INCORPORATING THE AUSTRALIAN ASSOCIATION OF AUT*'

In [0]:
acn = F.col("acn")

# ACN check digit: weights 8..1 on the first 8 digits; check = (10 - sum % 10) % 10 must equal the 9th digit
weights = [8, 7, 6, 5, 4, 3, 2, 1]
weighted_sum = sum(F.substring(acn, i + 1, 1).cast("int") * w for i, w in enumerate(weights))
check_ok = ((10 - (weighted_sum % 10)) % 10) == F.substring(acn, 9, 1).cast("int")

acn_profile = df.agg(
    F.count("*").alias("rows"),
    F.sum(acn.isNull().cast("int")).alias("null"),
    F.sum((acn != F.trim(acn)).cast("int")).alias("leading_trailing_spaces"),
    F.sum(acn.rlike(r"\D").cast("int")).alias("non_digit_chars"),
    F.sum((F.length(acn) != 9).cast("int")).alias("not_9_chars"),
    F.sum(acn.rlike(r"^[0-9]{9}$").cast("int")).alias("valid_9_digit_format"),
    F.sum(acn.startswith("0").cast("int")).alias("leading_zero"),
    F.sum((acn.rlike(r"^[0-9]{9}$") & ~check_ok).cast("int")).alias("check_digit_fails"),
    F.countDistinct(acn).alias("distinct_acns"),
).first().asDict()

for k, v in acn_profile.items():
    print(f"{k:26} {v:,}")

# Length distribution (anything other than 9 is a problem)
display(df.groupBy(F.length(acn).alias("acn_length")).count().orderBy("acn_length"))

# Check-digit failures by company type (RACN rows hold ARBNs, which may follow different rules)
display(df.filter(acn.rlike(r"^[0-9]{9}$") & ~check_ok)
          .groupBy("company_type", "current_name_indicator").count()
          .orderBy(F.desc("count")))

In [0]:
ct = F.col("company_type")

# 1) Every value with counts: all rows vs current-name rows only (what reaches Silver)
display(df.groupBy(ct.alias("company_type"))
          .agg(F.count("*").alias("all_rows"),
               F.sum((F.col("current_name_indicator") == "Y").cast("int")).alias("current_name_rows"))
          .orderBy(F.desc("all_rows")))

# 2) Formatting checks
display(df.agg(
    F.sum(ct.isNull().cast("int")).alias("null"),
    F.sum((ct != F.trim(ct)).cast("int")).alias("leading_trailing_spaces"),
    F.sum(ct.rlike(r"[a-z]").cast("int")).alias("lowercase"),
    F.sum((F.length(F.trim(ct)) != 4).cast("int")).alias("not_4_chars"),
    F.sum((~F.trim(ct).isin(list(TYPE_LABELS.keys()))).cast("int")).alias("not_in_dictionary"),
))

# 3) Consistency: type vs class, current-name rows only
display(df.filter(F.col("current_name_indicator") == "Y")
          .groupBy("company_type", "company_class").count()
          .orderBy("company_type", F.desc("count")))

In [0]:
cur = F.col("current_name_indicator") == "Y"
cc, cs = F.col("company_class"), F.col("company_sub_class")

# 1) Formatting + dictionary checks for both columns (all rows)
def fmt_checks(col, name, mapping):
    return [
        F.sum(col.isNull().cast("int")).alias(f"{name}_null"),
        F.sum((col != F.trim(col)).cast("int")).alias(f"{name}_spaces"),
        F.sum(col.rlike(r"[a-z]").cast("int")).alias(f"{name}_lowercase"),
        F.sum((F.length(F.trim(col)) != 4).cast("int")).alias(f"{name}_not_4_chars"),
        F.sum((col.isNotNull() & ~F.trim(col).isin(list(mapping.keys()))).cast("int")).alias(f"{name}_not_in_dictionary"),
    ]
checks = df.agg(*fmt_checks(cc, "class", CLASS_LABELS), *fmt_checks(cs, "sub_class", SUB_CLASS_LABELS)).first().asDict()
for k, v in checks.items():
    print(f"{k:28} {v:,}")

# 2) Value counts: all rows vs current-name rows (what reaches Silver)
for col_name in ["company_class", "company_sub_class"]:
    display(df.groupBy(col_name)
              .agg(F.count("*").alias("all_rows"),
                   F.sum(cur.cast("int")).alias("current_name_rows"))
              .orderBy(F.desc("current_name_rows")))

# 3) Codes not in the dictionary (if any), with examples
display(df.filter((cc.isNotNull() & ~cc.isin(list(CLASS_LABELS))) |
                  (cs.isNotNull() & ~cs.isin(list(SUB_CLASS_LABELS))))
          .groupBy("company_type", "company_class", "company_sub_class").count())

# 4) Consistency: type × class × sub-class combinations (current-name rows)
display(df.filter(cur)
          .groupBy("company_type", "company_class", "company_sub_class").count()
          .orderBy("company_type", "company_class", F.desc("count")))

# 5) Not-for-profit question: public companies limited by guarantee, by sub-class
display(df.filter(cur & (F.col("company_type") == "APUB") & (cc == "LMGT"))
          .groupBy("company_sub_class").count().orderBy(F.desc("count")))

In [0]:
cur = F.col("current_name_indicator") == "Y"
st  = F.col("status_code")
dereg_present = F.col("deregistration_date").isNotNull()     # still text at this point

# 1) Formatting + dictionary check
display(df.agg(
    F.sum(st.isNull().cast("int")).alias("null"),
    F.sum((st != F.trim(st)).cast("int")).alias("spaces"),
    F.sum(st.rlike(r"[a-z]").cast("int")).alias("lowercase"),
    F.sum((F.length(F.trim(st)) != 4).cast("int")).alias("not_4_chars"),
    F.sum((st.isNotNull() & ~st.isin(list(STATUS_LABELS))).cast("int")).alias("not_in_dictionary"),
))

# 2) Status counts: all rows vs current-name rows, and how many have a deregistration date
display(df.groupBy(st.alias("status_code"))
          .agg(F.count("*").alias("all_rows"),
               F.sum(cur.cast("int")).alias("current_name_rows"),
               F.sum((cur & dereg_present).cast("int")).alias("current_with_dereg_date"))
          .orderBy(F.desc("current_name_rows")))

# 3) Status vs deregistration date conflicts on current-name rows (were 10 in the 8 Oct profiling)
display(df.filter(cur & (st != "DRGD") & dereg_present)
          .select("acn", "company_name", "status_code", "registration_date", "deregistration_date"))

# 4) ACNs with no current-name row, by status (must not include active companies)
has_y = F.max(F.coalesce((F.col("current_name_indicator") == "Y").cast("int"), F.lit(0))).alias("has_y")
no_y  = df.groupBy("acn").agg(has_y).filter("has_y = 0")
display(df.join(no_y, "acn").groupBy("status_code").agg(F.countDistinct("acn").alias("acns_without_y_row")))

# 5) Preview: status mix by sub-class (current-name rows) — how segments differ
display(df.filter(cur)
          .groupBy("company_sub_class")
          .pivot("status_code", ["REGD", "DRGD", "SOFF", "EXAD", "NOAC", "CNCL", "DISS"])
          .count().na.fill(0)
          .orderBy(F.desc("REGD")))

In [0]:
cur  = F.col("current_name_indicator") == "Y"
snap = F.col("_snapshot_date")

def parsed(c):
    return F.expr(f"try_to_timestamp({c}, 'dd/MM/yyyy')").cast("date")

reg, dereg = parsed("registration_date"), parsed("deregistration_date")
fmt = r"^\d{2}/\d{2}/\d{4}$"

# 1) Format, parse and logic checks (current-name rows = what reaches Silver)
display(df.filter(cur).agg(
    F.sum(F.col("registration_date").isNull().cast("int")).alias("reg_null"),
    F.sum((F.col("registration_date").isNotNull() & ~F.col("registration_date").rlike(fmt)).cast("int")).alias("reg_bad_format"),
    F.sum((F.col("registration_date").isNotNull() & reg.isNull()).cast("int")).alias("reg_unparseable"),
    F.sum((reg > snap).cast("int")).alias("reg_after_snapshot"),
    F.min(reg).alias("reg_min"), F.max(reg).alias("reg_max"),
    F.sum((F.col("deregistration_date").isNotNull() & ~F.col("deregistration_date").rlike(fmt)).cast("int")).alias("dereg_bad_format"),
    F.sum((F.col("deregistration_date").isNotNull() & dereg.isNull()).cast("int")).alias("dereg_unparseable"),
    F.sum((dereg > snap).cast("int")).alias("dereg_after_snapshot"),
    F.sum((dereg < reg).cast("int")).alias("dereg_before_reg"),
    F.sum((dereg == reg).cast("int")).alias("dereg_same_day_as_reg"),
))

# 2) The 4 (or so) missing registration dates
display(df.filter(cur & F.col("registration_date").isNull())
          .select("acn", "company_name", "company_type", "status_code", "registration_date"))

# 3) Registrations per year (sanity check + suspicious spikes)
display(df.filter(cur).groupBy(F.year(reg).alias("reg_year")).count().orderBy("reg_year"))

# 4) Placeholder-date check: are any single dates suspiciously common?
display(df.filter(cur).groupBy(reg.alias("registration_date")).count()
          .orderBy(F.desc("count")).limit(10))

# 5) Deregistrations per month within the window (DRGD only)
display(df.filter(cur & (F.col("status_code") == "DRGD"))
          .groupBy(F.date_format(dereg, "yyyy-MM").alias("dereg_month")).count().orderBy("dereg_month"))

In [0]:
cur = df.filter(F.col("current_name_indicator") == "Y")

# 1) previous_state: what values appear, and how often
display(cur.groupBy("previous_state").count().orderBy(F.desc("count")))

# 2) state_registration_number: fill rate, and whether it only appears with previous_state
display(cur.agg(
    F.count_if(F.col("state_registration_number").isNotNull()).alias("srn_filled"),
    F.count_if(F.col("previous_state").isNotNull()).alias("prev_state_filled"),
    F.count_if(F.col("state_registration_number").isNotNull() & F.col("previous_state").isNull()).alias("srn_without_state"),
    F.count_if(F.col("state_registration_number").isNull() & F.col("previous_state").isNotNull()).alias("state_without_srn"),
    F.count_if(F.col("state_registration_number").rlike(r"[^0-9]")).alias("srn_non_numeric"),
))
display(cur.filter(F.col("state_registration_number").isNotNull())
           .select("acn", "company_name", "registration_date", "previous_state", "state_registration_number")
           .limit(20))

# 3) modified_since_last_report: what values appear
display(cur.groupBy("modified_since_last_report").count().orderBy(F.desc("count")))

# 4) abn: blanks, zeros, format, check digit, duplicates
abn = F.col("abn")
abn_digits = [F.substring(abn, i, 1).cast("int") for i in range(1, 12)]
weights = [10, 1, 3, 5, 7, 9, 11, 13, 15, 17, 19]
abn_sum = sum(((abn_digits[0] - 1) if i == 0 else abn_digits[i]) * w for i, w in enumerate(weights))

display(cur.agg(
    F.count_if(abn.isNull()).alias("abn_null"),
    F.count_if(abn == "0").alias("abn_zero"),
    F.count_if(abn.isNotNull() & (abn != "0") & ~abn.rlike(r"^[0-9]{11}$")).alias("abn_bad_format"),
    F.count_if(abn.rlike(r"^[0-9]{11}$") & (abn_sum % 89 != 0)).alias("abn_bad_checksum"),
    F.count_if(abn.rlike(r"^[0-9]{11}$") & (F.substring(abn, 3, 9) == F.col("acn"))).alias("abn_contains_acn"),
))

# ABNs shared by more than one company
display(cur.filter(abn.rlike(r"^[0-9]{11}$"))
           .groupBy("abn").agg(F.count("*").alias("companies"))
           .filter("companies > 1").orderBy(F.desc("companies")).limit(20))

# ABN fill rate by status (is a missing ABN linked to distress?)
display(cur.groupBy("status_code").agg(
    F.count("*").alias("companies"),
    F.round(F.avg((abn.rlike(r"^[0-9]{11}$")).cast("int")) * 100, 1).alias("pct_with_abn"),
).orderBy(F.desc("companies")))

In [0]:
cur = df.filter(F.col("current_name_indicator") == "Y")
reg = F.expr("try_to_timestamp(registration_date, 'dd/MM/yyyy')").cast("date")

# a) Why SA / WA / NT appear twice (hex shows hidden characters)
display(cur.filter(F.col("previous_state").isNotNull())
    .groupBy("previous_state", F.length("previous_state").alias("len"), F.hex("previous_state").alias("hex"))
    .count().orderBy("previous_state"))

# b) Do previous_state companies pre-date 1991? (NFK expected later)
display(cur.filter(F.col("previous_state").isNotNull())
    .groupBy(F.trim("previous_state").alias("prev_state"))
    .agg(F.count("*").alias("n"), F.min(reg).alias("min_reg"), F.max(reg).alias("max_reg"),
         F.count_if(reg >= F.lit("1991-01-01")).alias("reg_from_1991"))
    .orderBy(F.desc("n")))

# c) The 266 ABNs not built from the company's ACN
display(cur.filter((F.col("abn") != "0") & (F.substring("abn", 3, 9) != F.col("acn")))
    .select("acn", "company_name", "company_type", "company_sub_class", "status_code", "registration_date", "abn")
    .limit(30))

# d) ABN rate by sub-class (tests the trading-vs-dormant idea)
display(cur.groupBy("company_sub_class").agg(
    F.count("*").alias("companies"),
    F.round(F.avg((F.col("abn") != "0").cast("int")) * 100, 1).alias("pct_with_abn"))
  .orderBy(F.desc("companies")))

# Write Into Silver Table